# dynadiff-vlbi Phase 2.2 quickstart

This notebook runs a lightweight residual-refinement experiment where the verified baseline 3D U-Net provides the main reconstruction backbone and a compact visibility-guided branch predicts a residual correction with uncertainty support.

In [ ]:
import os
from pathlib import Path

candidate_roots = [
    Path.cwd(),
    Path('/content/dynadiff-vlbi'),
    Path('/content/DynaDiff-VLBI'),
]
repo_root = None
for candidate in candidate_roots:
    if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError('Could not find the dynadiff-vlbi repository root. Clone or upload the repo first.')
os.chdir(repo_root)
print(f'Using repo root: {repo_root.resolve()}')


In [ ]:
%pip install -e .


In [ ]:
!python scripts/generate_toy_dataset.py --base-config configs/phase2_residual_refine_default32.yaml --preset smoke
!python scripts/train_baseline.py --preset smoke --data-dir data/generated/smoke_phase2_residual_refine_default32 --run-name colab_phase22_baseline_smoke --epochs 1
!python scripts/train_baseline.py --base-config configs/phase2_visibility_default32.yaml --preset smoke --data-dir data/generated/smoke_phase2_residual_refine_default32 --run-name colab_phase22_visibility_smoke --epochs 1
!python scripts/train_baseline.py --base-config configs/phase2_residual_refine_default32.yaml --preset smoke --data-dir data/generated/smoke_phase2_residual_refine_default32 --run-name colab_phase22_residual_smoke --epochs 2 --backbone-checkpoint outputs/colab_phase22_baseline_smoke/checkpoints/best.pt
!python scripts/evaluate_model.py --base-config configs/phase2_residual_refine_default32.yaml --preset smoke --data-dir data/generated/smoke_phase2_residual_refine_default32 --run-name colab_phase22_residual_smoke --reference-baseline-checkpoint outputs/colab_phase22_baseline_smoke/checkpoints/best.pt --reference-visibility-checkpoint outputs/colab_phase22_visibility_smoke/checkpoints/best.pt


In [ ]:
import json
from pathlib import Path

from IPython.display import display
from PIL import Image

run_name = 'colab_phase22_residual_smoke'
summary_path = Path('outputs') / run_name / 'logs' / 'evaluation_summary.json'
figure_paths = sorted((Path('outputs') / run_name / 'figures').glob('*.png'))
print(json.dumps(json.loads(summary_path.read_text()), indent=2))
display(Image.open(figure_paths[0]))
